# Propensity Score Matching — Q4 2025 Holiday Promotion

**Goal**: Estimate the true causal effect of the promotion by matching each promoted store to a similar non-promoted store, removing selection bias.

**Why PSM?** The merchandising team selected stores based on size, revenue, and demographics. PSM "re-balances" the groups by pairing promoted stores with comparable controls.

**Covariates**: store_size, avg_weekly_revenue, competitor_density, median_household_income, foot_traffic_index

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from data.data_loader import load_panel_data, load_store_data
from data.feature_engineering import create_pre_treatment_features
from causal.propensity_score import (
    estimate_propensity_scores, match_nearest_neighbor,
    assess_balance, run_psm
)
from utils.visualization import plot_propensity_distribution, plot_balance

stores = load_store_data()
panel = load_panel_data()

# Build analysis dataset: store features + pre-campaign revenue stats
pre_features = create_pre_treatment_features(panel, pre_period_weeks=13)
analysis_df = stores.merge(pre_features, on='store_id')
print(f'Analysis dataset: {len(analysis_df)} stores, {analysis_df.shape[1]} features')

## Step 1: Estimate Propensity Scores

Logistic regression: P(Promoted | store features)

We need **overlap** — both groups should have similar propensity score ranges. If promoted stores all have PS > 0.8, there are no comparable controls to match.

In [ ]:
covariates = ['store_size', 'avg_weekly_revenue', 'competitor_density',
              'median_household_income', 'foot_traffic_index']

ps = estimate_propensity_scores(analysis_df, 'treated', covariates)
analysis_df['propensity_score'] = ps

ps_treated = ps[analysis_df['treated'] == 1]
ps_control = ps[analysis_df['treated'] == 0]

fig = plot_propensity_distribution(ps_treated, ps_control)
plt.show()

print(f'Promoted PS range:  [{ps_treated.min():.3f}, {ps_treated.max():.3f}]')
print(f'Control PS range:   [{ps_control.min():.3f}, {ps_control.max():.3f}]')
print(f'Good overlap region exists — matching is feasible.')

## Step 2: Match Promoted Stores to Similar Controls

1:1 nearest-neighbor matching within caliper of 0.05. Stores without a close match are dropped.

In [ ]:
matches = match_nearest_neighbor(ps, analysis_df['treated'].values, caliper=0.05)

n_promoted = (analysis_df['treated'] == 1).sum()
print(f'Promoted stores:     {n_promoted}')
print(f'Successfully matched: {len(matches)} ({len(matches)/n_promoted:.0%})')
print(f'Dropped (no match):  {n_promoted - len(matches)}')

# Show sample match quality
print('\nSample matched pairs (propensity score):')
for t_idx, c_idx in matches[:5]:
    t_id = analysis_df.iloc[t_idx]['store_id']
    c_id = analysis_df.iloc[c_idx]['store_id']
    print(f'  {t_id} (PS={ps[t_idx]:.4f}) <-> {c_id} (PS={ps[c_idx]:.4f}), gap={abs(ps[t_idx]-ps[c_idx]):.4f}')

## Step 3: Covariate Balance Check

After matching, the promoted and control groups should look statistically similar on ALL covariates. We check Standardized Mean Difference (SMD) — target is < 0.1 for each variable.

In [ ]:
# Before matching (raw imbalance)
balance_before = assess_balance(analysis_df, covariates, 'treated')
print('=== Before Matching ===')
print(balance_before[['covariate', 'std_mean_diff', 'balanced']].to_string(index=False))

print()

# After matching (should be balanced)
balance_after = assess_balance(analysis_df, covariates, 'treated', matches)
print('=== After Matching ===')
print(balance_after[['covariate', 'std_mean_diff', 'balanced']].to_string(index=False))

fig = plot_balance(balance_after)
plt.show()

## Step 4: Estimate the Causal Effect (ATT)

In [ ]:
result = run_psm(analysis_df, 'pre_avg_revenue', 'treated', covariates, caliper=0.05)

print('=' * 50)
print('PROPENSITY SCORE MATCHING — CAUSAL ESTIMATE')
print('Campaign: PROMO-2025-Q4-HOLIDAY')
print('=' * 50)
print(f'ATT (avg treatment effect):  {result.att:,.2f}')
print(f'Standard error:              {result.se:,.2f}')
print(f'95% confidence interval:     [{result.ci_lower:,.2f}, {result.ci_upper:,.2f}]')
print(f'p-value:                     {result.p_value:.4f}')
print(f'Matched store pairs:         {result.n_matched}')
print(f'Statistically significant:   {"YES" if result.p_value < 0.05 else "NO"}')
print()
print('Interpretation: After removing selection bias via matching,')
print('the estimated promotion effect is substantially lower than')
print('the naive 15% comparison reported by merchandising.')